In [1]:
import os

import pandas as pd

REPO_ROOT = os.path.dirname(os.getcwd())
DATA_PATH = os.path.join(REPO_ROOT, "dataset", "labels", "eligibility_annotations.csv")

df = pd.read_csv(DATA_PATH)
df.shape

(302, 6)

In [2]:
df["label"].value_counts()

label
OTHER_ELIGIBILITY    71
DISCIPLINE           61
NONE                 59
RESIDENCE            33
CAREER_STAGE         21
NATIONALITY          17
STUDENT_STATUS       17
EDUCATION            12
AGE                  11
Name: count, dtype: int64

In [3]:
df.groupby("opportunity_id").size().sort_values(ascending=False).head(10)

opportunity_id
mondriaan_fund_e52bef22781c                    22
national_culture_fund_bulgaria_8fca12fac738    17
kulturdirektoratet_norway_e1b48f36972d         16
dansateliers_rotterdam_e3404a6adf78            11
dgartes_portugal_111bdc1e2ebf                  10
mondriaan_fund_a939eddff305                     9
creative_ireland_e896d7db1d6a                   9
dansateliers_rotterdam_33931cad7e6f             8
kunsten_be_flanders_217b1f327dad                8
kunst_dk_denmark_823c0bbbc582                   8
dtype: int64

In [4]:
# 5 folds, grouped by opportunity so no call's chunks split across train/test
from sklearn.model_selection import StratifiedGroupKFold

X = df["chunk_text"]
y = df["label"]
groups = df["opportunity_id"]

# needs both: GroupKFold alone ignores the 9-class imbalance, StratifiedKFold alone
# would leak an opportunity's shared phrasing across train/test
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
all_labels = set(y.unique())

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
    train_groups = set(groups.iloc[train_idx])   # distinct opportunities in train
    test_groups = set(groups.iloc[test_idx])     # distinct opportunities in test
    overlap = train_groups & test_groups          # opportunities present on both sides
    assert not overlap, f"fold {fold}: group leakage on {overlap}"

    # smallest classes (AGE=11, EDUCATION=12) may still miss a fold's test split
    missing = all_labels - set(y.iloc[test_idx])
    print(f"fold {fold}: train={len(train_idx):3d}  test={len(test_idx):3d}  "
          f"missing_from_test={sorted(missing) or 'none'}")

fold 0: train=243  test= 59  missing_from_test=none
fold 1: train=241  test= 61  missing_from_test=none
fold 2: train=240  test= 62  missing_from_test=none
fold 3: train=243  test= 59  missing_from_test=['STUDENT_STATUS']
fold 4: train=241  test= 61  missing_from_test=none


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

X_text = df[["chunk_text"]]  # DataFrame, not Series — ColumnTransformer needs a column name to select

baseline = Pipeline([
    # single-column selector: ColumnTransformer hands TfidfVectorizer a 1D array when given a bare column name
    ("features", ColumnTransformer([("tfidf", TfidfVectorizer(), "chunk_text")])),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

param_grid = {
    "features__tfidf__ngram_range": [(1, 1), (1, 2)],
    "features__tfidf__min_df": [1, 2, 3],
    "clf__C": [0.1, 1, 10],
}

# scoring=f1_macro, not accuracy — every class weighted equally despite the 6.5x imbalance
grid = GridSearchCV(baseline, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
grid.fit(X_text, y, groups=groups)

print("best params:", grid.best_params_)
print("best macro-F1 (mean over folds):", grid.best_score_)

best params: {'clf__C': 1, 'features__tfidf__min_df': 1, 'features__tfidf__ngram_range': (1, 1)}
best macro-F1 (mean over folds): 0.5894088051513944


In [7]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_predict

# out-of-fold predictions for the whole corpus, using the best hyperparameters found above
y_pred = cross_val_predict(grid.best_estimator_, X_text, y, cv=cv, groups=groups, n_jobs=-1)

print(classification_report(y, y_pred))

# ANNOTATION_GUIDELINES.md §2 menu order, for a readable confusion matrix
LABEL_ORDER = ["RESIDENCE", "NATIONALITY", "AGE", "DISCIPLINE", "CAREER_STAGE",
               "EDUCATION", "STUDENT_STATUS", "OTHER_ELIGIBILITY", "NONE"]

cm = confusion_matrix(y, y_pred, labels=LABEL_ORDER)
pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER)

                   precision    recall  f1-score   support

              AGE       1.00      0.91      0.95        11
     CAREER_STAGE       0.84      0.76      0.80        21
       DISCIPLINE       0.50      0.57      0.53        61
        EDUCATION       0.62      0.42      0.50        12
      NATIONALITY       0.50      0.41      0.45        17
             NONE       0.39      0.37      0.38        59
OTHER_ELIGIBILITY       0.59      0.58      0.58        71
        RESIDENCE       0.55      0.70      0.61        33
   STUDENT_STATUS       0.85      0.65      0.73        17

         accuracy                           0.56       302
        macro avg       0.65      0.60      0.62       302
     weighted avg       0.57      0.56      0.56       302



,RESIDENCE,NATIONALITY,AGE,DISCIPLINE,CAREER_STAGE,EDUCATION,STUDENT_STATUS,OTHER_ELIGIBILITY,NONE
RESIDENCE,23,1,0,3,0,0,0,2,4
NATIONALITY,1,7,0,5,0,0,0,1,3
AGE,0,0,10,0,0,0,0,0,1
DISCIPLINE,5,3,0,35,0,1,0,8,9
CAREER_STAGE,0,0,0,5,16,0,0,0,0
EDUCATION,0,0,0,1,2,5,2,1,1
STUDENT_STATUS,0,1,0,0,0,2,11,0,3
OTHER_ELIGIBILITY,7,1,0,9,0,0,0,41,13
NONE,6,1,0,12,1,0,0,17,22


In [8]:
# the confusion matrix's densest error cluster: DISCIPLINE / OTHER_ELIGIBILITY / NONE mixing with each other
confused = {"NONE", "DISCIPLINE", "OTHER_ELIGIBILITY"}
errors = df.assign(pred=y_pred)
errors = errors[(errors["label"] != errors["pred"])
                & errors["label"].isin(confused) & errors["pred"].isin(confused)]

for (true_label, pred_label), group in errors.groupby(["label", "pred"]):
    print(f"\n=== true={true_label}  predicted={pred_label}  (n={len(group)}) ===")
    for text in group["chunk_text"].head(3):
        print(f"  - {text}")


=== true=DISCIPLINE  predicted=NONE  (n=9) ===
  - Publishers with a Norwegian organization number may apply for this scheme.
  - Marta & Kim are searching for a total of 2 additional performers, who meet the following conditions:
  - We are inviting anyone to send us a proposal to create a performance, character or small happening around the (social) aspects of drinking tea (ceremonial, historical, colonial, its gossipy aspect etc) to be presented during our Dress-up-tea-Party.

=== true=DISCIPLINE  predicted=OTHER_ELIGIBILITY  (n=8) ===
  - The Rencontres d’Arles curatorial research grant is aimed at exhibition curators wishing to carry out an original project relating to image or photography.
  - For the ‘Dress-up’ part of our party, we welcome fashion- & costume designers to send a proposal of a selection of pieces/ costumes (from their oeuvre) to be used during the Dress-Up-Tea-Party.
  - Limitation is the Audiovisual sector (MEDIA).

=== true=NONE  predicted=DISCIPLINE  (n=12) =

In [9]:
pd.set_option("display.max_colwidth", 120)

# confident-error ranking: misclassifications where the model was most sure of a
# *different* label are the best candidates for a real annotation mistake, not
# just a hard/ambiguous chunk the model was unsure about
proba = cross_val_predict(grid.best_estimator_, X_text, y, cv=cv, groups=groups,
                           method="predict_proba", n_jobs=-1)
classes = grid.best_estimator_.classes_
confidence = proba.max(axis=1)
pred_label = classes[proba.argmax(axis=1)]

review = df.assign(pred=pred_label, confidence=confidence)
review = review[review["label"] != review["pred"]].sort_values("confidence", ascending=False)
review[["chunk_id", "chunk_text", "label", "pred", "confidence"]].head(20)

,chunk_id,chunk_text,label,pred,confidence
49,kunsten_be_flanders_217b1f327dad_1,You are pursuing a master’s degree or have just graduated as a musician,STUDENT_STATUS,EDUCATION,0.346432
178,fundacao_gulbenkian_86e33a273257_1,"Will be enrolled in the 2nd, 3rd or 4th year of a bachelor's degree or integrated master's programme, or in the 1st ...",STUDENT_STATUS,EDUCATION,0.314829
68,dgartes_portugal_111bdc1e2ebf_6,"Entities must frame their activity in only one application and, in the event of support being granted, the respectiv...",NONE,OTHER_ELIGIBILITY,0.277418
298,kunstlerhaus_bethanien_ad1d16a215d9_5a,You can be a foreign citizen and apply for our stipends and grants.,NATIONALITY,NONE,0.261745
38,kunsten_be_flanders_1c78c30f66d3_0,"Eligible for support are Dutch-Belgian co-projects across all disciplines, but special attention is given to the per...",NATIONALITY,DISCIPLINE,0.254239
171,u_jazdowski_warsaw_ea8781efb99b_0,"Open call for Palestinian artists , cultural practitioners , theorists , curators , and people working at the inters...",NATIONALITY,DISCIPLINE,0.253983
59,kunsten_be_flanders_df27ab545c60_2,"We especially welcome applications from artists with migration backgrounds, artists of color, queer, neurodiverse, a...",NONE,NATIONALITY,0.253039
210,kulturdirektoratet_norway_90a7593f042e_3,"Individual artists, artist groups and individuals acting as independent actors in the field should instead apply to ...",OTHER_ELIGIBILITY,DISCIPLINE,0.248098
43,kunsten_be_flanders_1c78c30f66d3_5,Note: only the Belgian partner can apply for this funding.,NATIONALITY,NONE,0.244675
243,kulturdirektoratet_norway_e1b48f36972d_1,"The concerts can take place in Norway and abroad, but we place particular emphasis on concerts in Norway.",NONE,RESIDENCE,0.239356
